# 0. Setup

In [1]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str, reindex_entity

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

In [ ]:
import ibis
from ibis import _

# 1. Define the spatial/temporal boundaries for the window
w = ibis.window(group_by="registered_number", order_by="year")

# 2. Extract columns you want to difference dynamically
cols_to_diff = ["gva1", "total_assets", "employees"]

# 3. Build a dictionary of safe-differencing expressions
diff_exprs = {}
for col in cols_to_diff:
    diff_exprs[f"D1_{col}"] = ibis.ifelse(
        # CONDITION: Is the previous row exactly one year ago?
        _.year == _.year.lag(1).over(w) + 1,
        # TRUE: Calculate the first difference
        _[col] - _[col].lag(1).over(w),
        # FALSE: Force a NULL (safe fallback for gaps and first-years)
        ibis.literal(None, type="float64")
    )

# 4. Execute the mutation in a single DuckDB pass
t_panel_diffed = t_panel.mutate(**diff_exprs)

# 1a. LMM
$$
\begin{align*}
y_{it} &= \alpha_i + \gamma_t + w_{it}\theta + \beta E[TFP_{-i,g,t}\vert{}g] + \epsilon_{it} \\
x_{it} &=
\begin{pmatrix}
k_{it} & l_{it}
\end{pmatrix}
\end{align*}
$$
**Run 1**
- ✅ Model 'peer3' estimated: peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
- ✅ Model 'peer3_no_firm_fe' estimated: peer_tfp_ttwa_donut=0.204, peer_tfp_pc4_donut=0.142, peer_tfp_pc8=0.559
- ✅ Model 'peer3_employees' estimated: ln_employees=0.009, peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
Panel regressions complete. 3 models, writing.

In [10]:
panel_name = "working_yearly_g"
out_name = "results_1a_lmm_pc8"

# t_diffed = ibis.read_parquet(dirs.output_dir / f"{panel_name}_diff.parquet", engine="pyarrow")
df_diff = pd.read_parquet(dirs.tmp_dir / f"{panel_name}_diff.parquet", engine="pyarrow")
df_diff['registered_number'] = df_diff.index.get_level_values('registered_number')
df_diff['year'] = df_diff.index.get_level_values('year')

tdumm_cols = []
if 't' in mod.fe:
    df_m1 = pd.get_dummies(df_diff['year'] - 1, prefix='year')
    # Multiply every value in df_m1 by -1 to create the difference
    df_m1_minus = df_m1 * -1
    df_m2 = pd.get_dummies(df_diff['year'], prefix='year')
    df_m_joined = df_m1_minus + df_m2
    df_diff = df_diff.join(df_m_joined, how='left')
    tdumm_cols = [col for col in df_diff.columns if col.startswith('year_')]
display(df_diff[tdumm_cols].head())

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models_1 = {
    'lmm_exog': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t'],
        'description': 'Strict exogeneity, structural form'
    },
    'lmm_instr': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l'
            ],
        },
        'fe': ['t'],
        'description': 'Instrument endogenous effect, LMM'
    }
}
models_3 = {
    'lmm_3_rings': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'W': ['wg2_y', 'wg2_k', 'wg2_l', 'wg3_l', 'wg3_k', 'wg3_y'],
        'fe': ['t'],
        'description': 'Strict exogeneity, structural form'
    },
    'lmm_3_rings_instr': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l']
        },
        'fe': ['t'],
        'description': '3-level regressors, squared instruments'
    },
    'lmm_3_rings_instr': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l', 'wg1wg2_k', 'wg1wg2_l', 'wg1wg3_k', 'wg1wg3_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l', 'wg2wg3_k', 'wg2wg3_l', 'wg2wg1_k', 'wg2wg1_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l', 'wg3wg1_k', 'wg3wg1_l', 'wg3wg2_k', 'wg3wg2_l']
        },
        'fe': ['t'],
        'description': '3-level regressors, many instruments'
    }
}

models = models_1

# 5m run for 3 complicated IV models
run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]

for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel((mod, df_diff, model_name))
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_name}.txt")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

year_2005  year_2006  year_2007  year_2008  year_2009  \
registered_number year                                                          
00000006          2012        NaN          0          0          0          0   
                  2013        NaN          0          0          0          0   
00000140          2006        NaN          1          0          0          0   
                  2007        NaN         -1          1          0          0   
                  2008        NaN          0         -1          1          0   

                        year_2010  year_2011  year_2012  year_2013  year_2014  \
registered_number year                                                          
00000006          2012          0         -1          1          0          0   
                  2013          0          0         -1          1          0   
00000140          2006          0          0          0          0          0   
                  2007          0          0          0          0          0   
                  2008          0          0          0          0          0   

                        year_2015  year_2016  year_2017  year_2018  year_2019  \
registered_number year                                                          
00000006          2012          0          0          0          0          0   
                  2013          0          0          0          0          0   
00000140          2006          0          0          0          0          0   
                  2007          0          0          0          0          0   
                  2008          0          0          0          0          0   

                        year_2020  year_2021  year_2022  year_2023  year_2024  
registered_number year                                                         
00000006          2012          0          0          0          0        NaN  
                  2013          0          0          0          0        NaN  
00000140          2006          0          0          0          0        NaN  
                  2007          0          0          0          0        NaN  
                  2008          0          0          0          0        NaN

Running model 'lmm_exog' as panel OLS
✅ Model 'lmm_exog' estimated: k=0.238, l=0.552, wg1_y=0.171, wg1_k=-0.050, wg1_l=-0.101
Running model 'lmm_instr' as linearmodels panel IV, formula: y ~ k + l + wg1_k + wg1_l + year_2007 + year_2008 + year_2009 + year_2010 + year_2011 + year_2012 + year_2013 + year_2014 + year_2015 + year_2016 + year_2017 + year_2018 + year_2019 + year_2020 + year_2021 + year_2022 + year_2023 + year_2024 + [wg1_y ~ w2g1_k + w2g1_l + w3g1_k + w3g1_l]
J-statistic (rej if overidentified): 3.43, p-value: 0.330
✅ Model 'lmm_instr' estimated: k=0.233, l=0.555, wg1_y=-0.384, wg1_k=0.118, wg1_l=0.249
Panel regressions complete. 2 models, writing to results_1a_lmm_pc8.txt


In [ ]:
# for args in worker_args:
#     mod, _, model_name = args
#     res, beta, effects_dict = run_panel(args)
#     if res is None:
#         print(f"Model '{model_name}' failed. Skipping.")
#         continue
#     run_res_series.append((res, mod, model_name))   # type: ignore

# model_count = len(run_res_series)
# if model_count:
#     output_str = "\n".join(map(format_str, run_res_series))
#     print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
#     with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
#         f.write(output_str)

# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [3]:
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str

panel_name = "working_yearly_n"
out_name = "results_2a_dd"

df_diff = pd.read_parquet(dirs.tmp_dir / f"{panel_name}_diff.parquet", engine="pyarrow")
df_diff['registered_number'] = df_diff.index.get_level_values('registered_number')
df_diff['year'] = df_diff.index.get_level_values('year')
print(df_diff.head())

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
m_models = {
    'dd1': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 1 model'
    },
    'dd2': {
        'Y': 'y',
        'X': ['k', 'l', 'wd2_y', 'wd2_k', 'wd2_l'],
        'Z': {
            'wd2_y': ['w2d2_k', 'w2d2_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 2 model'
    },
    'dd3': {
        'Y': 'y',
        'X': ['k', 'l', 'wd3_y', 'wd3_k', 'wd3_l'],
        'Z': {
            'wd3_y': ['w2d3_k', 'w2d3_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 3 model'
    }
}
iv_models = {
    'dd1-2lag': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'description': '2nd-order spatial lag model'
    },
    'dd1-3lag': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['i', 't'],
        'description': '3rd-order spatial lag model'
    }
}

models = m_models

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]

for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel((mod, df_diff, model_name))
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_name}.txt")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\lazyst\\Files\\ucl\\Dissertation\\model\\tmp\\working_yearly_n_diff.parquet'